# 02 - Recommendation Policy Evaluation

**Objective.** Execute the actual frozen recommendation stack on the original controlled scenarios and the predeclared 120-pair expanded design. Measure score, ranking, preferred-skill, final-activity, tier, compliance, coverage, completion, and invariant behavior.

**Inputs.** `scenario_definitions.json`, `expanded_config.json`, and the repository-relative C2 backend. Pair generation uses seed `20261022` and retains every generated pair.

In [1]:
from pathlib import Path
import pandas as pd
from evaluation_runner import evaluate_recommendation
from expanded_evaluation import evaluate_expanded

HERE = Path.cwd().resolve()
assert (HERE / 'evaluation_runner.py').is_file(), 'Run the notebook from the evaluation folder.'
SOURCE_BACKEND = HERE.parents[2]
SCENARIOS = HERE / 'scenario_definitions.json'
CONFIG = HERE / 'expanded_config.json'
print('Backend:', SOURCE_BACKEND.relative_to(SOURCE_BACKEND.parent))

Backend: backend


In [2]:
original = evaluate_recommendation(SOURCE_BACKEND, SCENARIOS)
history = original['overall']['history_sensitivity']
pd.DataFrame([
    {'experiment': 'original six pairs', 'measure': 'score changes', 'numerator': history['score_changes'], 'denominator': history['score_comparable_pairs'], 'rate': history['score_change_rate']},
    {'experiment': 'original six pairs', 'measure': 'final activity changes', 'numerator': history['recommendation_changes'], 'denominator': history['matched_pairs'], 'rate': history['recommendation_change_rate']},
    {'experiment': 'original controlled scenarios', 'measure': 'curriculum compliance', 'numerator': original['overall']['curriculum_valid'], 'denominator': original['overall']['recommendations'], 'rate': original['overall']['curriculum_compliance_rate']},
    {'experiment': 'original controlled scenarios', 'measure': 'tier-policy compliance', 'numerator': original['overall']['tier_policy_valid'], 'denominator': original['overall']['recommendations'], 'rate': original['overall']['tier_policy_compliance_rate']},
])

,experiment,measure,numerator,denominator,rate
0,original six pairs,score changes,5,6,0.833333
1,original six pairs,final activity changes,1,6,0.166667
2,original controlled scenarios,curriculum compliance,18,18,1.000000
3,original controlled scenarios,tier-policy compliance,18,18,1.000000


In [3]:
expanded = evaluate_expanded(SOURCE_BACKEND, CONFIG, SCENARIOS)
m = expanded['metrics']
metric_names = [
    ('Final activity change','activity_changes','history_pairs','activity_change_rate'),
    ('Equal-eligibility activity change','equal_eligibility_activity_changes','equal_eligibility_pairs','equal_eligibility_activity_change_rate'),
    ('XGBoost score change','score_changes','history_pairs','score_change_rate'),
    ('Candidate-ranking change','ranking_changes','history_pairs','ranking_change_rate'),
    ('Preferred-skill change','preferred_skill_changes','history_pairs','preferred_skill_change_rate'),
    ('Tier change','tier_changes','history_pairs','tier_change_rate'),
    ('Curriculum compliance','curriculum_valid','scenario_decisions','curriculum_compliance_rate'),
    ('Tier-policy compliance','tier_valid','scenario_decisions','tier_policy_compliance_rate'),
    ('Invariant stability','invariant_stable','invariant_pairs','invariant_stability_rate'),
]
pd.DataFrame([{'measure': label, 'numerator': m[num], 'denominator': m[den], 'rate': m[value]} for label,num,den,value in metric_names])

,measure,numerator,denominator,rate
0,Final activity change,22,120,0.183333
1,Equal-eligibility activity change,17,96,0.177083
2,XGBoost score change,111,120,0.925000
3,Candidate-ranking change,75,120,0.625000
4,Preferred-skill change,65,120,0.541667
5,Tier change,2,120,0.016667
6,Curriculum compliance,280,280,1.000000
7,Tier-policy compliance,280,280,1.000000
8,Invariant stability,20,20,1.000000


In [4]:
agreement = pd.DataFrame([
    {'component': 'prediction direction', 'numerator': m['direction_expectations_met'], 'denominator': m['direction_expectations_total'], 'agreement': m['direction_expectation_agreement']},
    {'component': 'coverage', 'numerator': m['coverage_expectations_met'], 'denominator': m['coverage_expectations_total'], 'agreement': m['coverage_expectation_agreement']},
    {'component': 'invariance', 'numerator': m['invariant_stable'], 'denominator': m['invariant_pairs'], 'agreement': m['invariant_stability_rate']},
    {'component': 'combined Policy-Expectation Agreement', 'numerator': m['policy_expectations_met'], 'denominator': m['policy_expectations_total'], 'agreement': m['policy_expectation_agreement']},
])
display(agreement)
display(pd.DataFrame(expanded['stratified_results']))
print('Mean absolute probability difference:', round(m['mean_absolute_probability_difference'], 4))
print('All unequal candidate sets differ only by Tier B activities:', m['all_eligibility_differences_tier_b_only'])

,component,numerator,denominator,agreement
0,prediction direction,98,120,0.816667
1,coverage,280,280,1.000000
2,invariance,20,20,1.000000
3,combined Policy-Expectation Agreement,398,420,0.947619


,stratum,value,pairs,activity_change_rate,score_change_rate,ranking_change_rate,tier_change_rate,direction_agreement,equal_eligibility_rate
0,grade,1,24,0.125000,1.000,0.500000,0.000000,0.833333,0.750000
1,grade,2,24,0.083333,1.000,0.666667,0.000000,0.916667,0.666667
2,grade,3,24,0.250000,0.875,0.666667,0.041667,0.833333,0.833333
3,grade,4,24,0.250000,0.875,0.583333,0.000000,0.750000,0.875000
4,grade,5,24,0.208333,0.875,0.708333,0.041667,0.750000,0.875000
5,stage,early,40,0.000000,0.850,0.275000,0.000000,0.625000,1.000000
6,stage,late,40,0.125000,1.000,0.825000,0.050000,1.000000,0.425000
7,stage,middle,40,0.425000,0.925,0.775000,0.000000,0.825000,0.975000
8,contrast,improving_vs_declining,30,0.066667,1.000,0.600000,0.000000,0.933333,0.900000
9,contrast,recent_correct_vs_recent_incorrect,30,0.133333,1.000,0.566667,0.000000,0.766667,0.833333


Mean absolute probability difference: 0.2467
All unequal candidate sets differ only by Tier B activities: True


In [5]:
display(pd.DataFrame(expanded['original_six_pair_diagnosis']))
display(pd.DataFrame(original['baseline_session_comparison']).T)

,pair_id,activity_a,activity_b,game_a,game_b,score_a,score_b,activity_changed,eligible_sets_equal,coverage_set_a,coverage_set_b,model_preferred_a,model_preferred_b,reason_a,reason_b,explanation
0,EXTENDED-HISTORY,ORD_A_01,ORD_A_01,ORDERING,ORDERING,0.801814,0.349830,False,True,ORDERING,ORDERING,ADDITION,PLACE_VALUE,MODEL_RANKED_COVERAGE_CANDIDATE,MODEL_RANKED_COVERAGE_CANDIDATE,"Probability changed, but a singleton foundatio..."
1,G1-PERFORMANCE,CMP_A_01,ORD_A_01,COMPARISON,ORDERING,0.895539,0.347975,True,False,ADDITION|COMPARISON|ORDERING|SUBTRACTION,ORDERING,COMPARISON,ORDERING,MINIMUM_TOTAL_BUDGET_ALLOCATION,MODEL_RANKED_COVERAGE_CANDIDATE,The policy state changed from MINIMUM_TOTAL_BU...
2,G2-PERFORMANCE,CMP_A_01,CMP_A_01,COMPARISON,COMPARISON,0.860204,0.270358,False,False,COMPARISON|ORDERING,COMPARISON|ORDERING,COMPARISON,ADDITION,MODEL_RANKED_COVERAGE_CANDIDATE,MODEL_RANKED_COVERAGE_CANDIDATE,"The global model preference changed, but ADDIT..."
3,G3-PERFORMANCE,PV_A_01,PV_A_01,PLACE_VALUE,PLACE_VALUE,0.860204,0.258679,False,False,ADDITION|COMPARISON|DIVISION|ORDERING|PLACE_VA...,ADDITION|COMPARISON|DIVISION|ORDERING|PLACE_VA...,COMPARISON,MULTIPLICATION,MODEL_RANKED_COVERAGE_CANDIDATE,MODEL_RANKED_COVERAGE_CANDIDATE,"The global model preference changed, but MULTI..."
4,G5-TREND,CNT_A_01,CNT_A_01,COUNTING,COUNTING,0.694323,0.344236,False,True,ADDITION|COMPARISON|COUNTING|MULTIPLICATION|OR...,ADDITION|COMPARISON|COUNTING|MULTIPLICATION|OR...,COUNTING,COUNTING,MODEL_RANKED_COVERAGE_CANDIDATE,MODEL_RANKED_COVERAGE_CANDIDATE,"Probability changed, but the model-preferred e..."
5,SKILL-SPECIFIC,ORD_A_01,ORD_A_01,ORDERING,ORDERING,0.604023,0.604023,False,False,ADDITION|MULTIPLICATION|ORDERING|PLACE_VALUE|S...,ADDITION|MULTIPLICATION|ORDERING|PLACE_VALUE|S...,ORDERING,ORDERING,MODEL_RANKED_COVERAGE_CANDIDATE,MODEL_RANKED_COVERAGE_CANDIDATE,"Probability changed, but the model-preferred e..."


,sessions,completed,valid_session_completion_rate,mean_valid_responses,mean_skill_coverage,curriculum_compliance_rate,tier_policy_compliance_rate,recommendations,trained_ml_usage_rate
fixed_sequence,15.0,5.0,0.333333,23.400000,1.0,1.0,1.0,351.0,0.000000
rule_only,15.0,15.0,1.000000,20.133333,1.0,1.0,1.0,302.0,0.000000
hybrid,15.0,15.0,1.000000,19.533333,1.0,1.0,1.0,293.0,0.948805


## Interpretation and limitations

The expanded experiment supports a bounded history-adaptation claim: scores and candidate ordering respond to synthetic historical differences, while coverage, curriculum, prerequisites, and tier rules deliberately limit final-task changes. Policy-Expectation Agreement measures internal specification agreement, not recommendation accuracy. Synthetic outcomes do not establish better learning or independent pedagogical appropriateness.